In [1]:
import pandas as pd

# Lendo as duas planilhas
janeiro = pd.read_excel('vendas_janeiro_2025_SUJO.xlsx')
fevereiro = pd.read_excel('vendas_fevereiro_2025_SUJO.xlsx')

# Juntando tudo numa tabela só
df = pd.concat([janeiro, fevereiro], ignore_index=True)

# Visualizando o tamanho
print(f"Total de linhas: {df.shape[0]}")
print(f"Total de colunas: {df.shape[1]}")

# Primeiras linhas
df.head()

Total de linhas: 300
Total de colunas: 11


,ID_VENDA,DATA_VENDA,PRODUTO,CATEGORIA,QUANTIDADE,PRECO_UNIT,VENDEDOR,PAGAMENTO,CIDADE,CPF_CLIENTE,NOTA_FISCAL
0,1,2025-01-08,geladeira frost free,Eletrodoméstico,2,"3431,89",João Santos,CARTAO DE CREDITO,São Paulo,NaN,88907.0
1,2,jan/21/2025,televisao 55,electrodomestico,NaN,"2547,02",Ana Souza,Cartão de Crédito,Salavdor,814.532.448-45,54118.0
2,3,NaN,GELADEIRA FROST FREE,Eletrodoméstico,3,NaN,Joao Santos,pix,S. Paulo,487.180.665-47,91070.0
3,4,03/01/2025,micro-ondas 30l,Eletronico,3,NaN,Carlos Silva,dinheiro,Rio,954.473.266-57,97841.0
4,5,06.01.2025,CELULAR SAMSUNG,Celular,NaN,R$1858.93,Pedro Costa,boleto,curitiba,432.963.886-17,14207.0


In [2]:
print("=== NULOS POR COLUNA ===")
print(df.isnull().sum())
print(f"\n=== TIPOS DE DADOS ===")
print(df.dtypes)

=== NULOS POR COLUNA ===
ID_VENDA         0
DATA_VENDA      43
PRODUTO          0
CATEGORIA        0
QUANTIDADE     131
PRECO_UNIT      83
VENDEDOR         0
PAGAMENTO        0
CIDADE           0
CPF_CLIENTE     57
NOTA_FISCAL     45
dtype: int64

=== TIPOS DE DADOS ===
ID_VENDA         int64
DATA_VENDA         str
PRODUTO            str
CATEGORIA          str
QUANTIDADE      object
PRECO_UNIT      object
VENDEDOR           str
PAGAMENTO          str
CIDADE             str
CPF_CLIENTE        str
NOTA_FISCAL    float64
dtype: object


In [3]:
# Fazendo uma cópia limpa pra não perder o original sujo
df_limpo = df.copy()

# 1. PADRONIZAR PRODUTO - tudo minúsculo e sem espaço duplo
df_limpo['PRODUTO'] = df_limpo['PRODUTO'].str.lower().str.strip()

# 2. PADRONIZAR CATEGORIA - tudo minúsculo e sem espaço
df_limpo['CATEGORIA'] = df_limpo['CATEGORIA'].str.lower().str.strip()

# 3. PADRONIZAR VENDEDOR - tudo título (primeira letra maiúscula) e sem espaço duplo
df_limpo['VENDEDOR'] = df_limpo['VENDEDOR'].str.title().str.strip().str.replace(r'\s+', ' ', regex=True)

# 4. PADRONIZAR PAGAMENTO - tudo minúsculo e sem espaço
df_limpo['PAGAMENTO'] = df_limpo['PAGAMENTO'].str.lower().str.strip()

# 5. PADRONIZAR CIDADE - tudo título e sem espaço
df_limpo['CIDADE'] = df_limpo['CIDADE'].str.title().str.strip()

# Verificando
print("ANTES:", df['VENDEDOR'].unique())
print("\nDEPOIS:", df_limpo['VENDEDOR'].unique())

ANTES: <StringArray>
[ 'João  Santos',    'Ana  Souza',   'Joao Santos', 'Carlos  Silva',
   'Pedro Costa',  'carlos silva',    'maria lima',   'JOAO SANTOS',
  'Pedro  Costa',     'ana souza',   'João Santos',    'MARIA LIMA',
   'Maria Limaa',   'PEDRO COSTA',     'Ana Sousa',  'Carlos Silva',
   'Maria  Lima',     'ANA SOUZA', 'Carlos Silvaa',   'pedro costa',
    'Maria Lima',  'CARLOS SILVA',   'Perdo Costa',     'Ana Souza',
   'joao santos']
Length: 25, dtype: str

DEPOIS: <StringArray>
[  'João Santos',     'Ana Souza',   'Joao Santos',  'Carlos Silva',
   'Pedro Costa',    'Maria Lima',   'Maria Limaa',     'Ana Sousa',
 'Carlos Silvaa',   'Perdo Costa']
Length: 10, dtype: str


In [4]:
# Corrigindo erros de digitação nos vendedores manualmente
correcoes_vendedor = {
    'Carlos Silvaa': 'Carlos Silva',
    'Maria Limaa': 'Maria Lima',
    'Perdo Costa': 'Pedro Costa',
    'Ana Sousa': 'Ana Souza',
    'Joao Santos': 'João Santos',
}

df_limpo['VENDEDOR'] = df_limpo['VENDEDOR'].replace(correcoes_vendedor)

print("Vendedores únicos após correção:")
print(df_limpo['VENDEDOR'].unique())
print(f"Total: {df_limpo['VENDEDOR'].nunique()} vendedores")

Vendedores únicos após correção:
<StringArray>
['João Santos', 'Ana Souza', 'Carlos Silva', 'Pedro Costa', 'Maria Lima']
Length: 5, dtype: str
Total: 5 vendedores


In [5]:
# Corrigindo cidades
correcoes_cidade = {
    'Salavdor': 'Salvador',
    'Curityba': 'Curitiba',
    'S. Paulo': 'São Paulo',
    'Sp': 'São Paulo',
    'Rio': 'Rio De Janeiro',
    'Rj': 'Rio De Janeiro',
    'Bh': 'Belo Horizonte',
    'B. Horizonte': 'Belo Horizonte',
    'Cwb': 'Curitiba',
    'Ssa': 'Salvador',
}

df_limpo['CIDADE'] = df_limpo['CIDADE'].replace(correcoes_cidade)

print("ANTES:", df['CIDADE'].nunique(), "versões")
print("DEPOIS:", df_limpo['CIDADE'].nunique(), "versões")
print(df_limpo['CIDADE'].unique())

ANTES: 25 versões
DEPOIS: 6 versões
<StringArray>
[     'São Paulo',       'Salvador', 'Rio De Janeiro',       'Curitiba',
      'Sao Paulo', 'Belo Horizonte']
Length: 6, dtype: str


In [6]:
# Corrigindo Sao Paulo sem acento
df_limpo['CIDADE'] = df_limpo['CIDADE'].replace({
    'Sao Paulo': 'São Paulo'
})

print(df_limpo['CIDADE'].unique())
print(f"Total: {df_limpo['CIDADE'].nunique()} cidades")

<StringArray>
['São Paulo', 'Salvador', 'Rio De Janeiro', 'Curitiba', 'Belo Horizonte']
Length: 5, dtype: str
Total: 5 cidades


In [7]:
# Corrigindo PRECO_UNIT — remover R$, trocar vírgula por ponto e converter pra número
df_limpo['PRECO_UNIT'] = (
    df_limpo['PRECO_UNIT']
    .astype(str)
    .str.replace('R$', '', regex=False)
    .str.replace('R$ ', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
)
df_limpo['PRECO_UNIT'] = pd.to_numeric(df_limpo['PRECO_UNIT'], errors='coerce')

# Corrigindo QUANTIDADE — remover " un", converter pra número
df_limpo['QUANTIDADE'] = (
    df_limpo['QUANTIDADE']
    .astype(str)
    .str.replace(' un', '', regex=False)
    .str.strip()
)
df_limpo['QUANTIDADE'] = pd.to_numeric(df_limpo['QUANTIDADE'], errors='coerce')

# Verificando
print("Tipos depois da correção:")
print(df_limpo[['QUANTIDADE', 'PRECO_UNIT']].dtypes)
print(f"\nNulos em PRECO_UNIT: {df_limpo['PRECO_UNIT'].isnull().sum()}")
print(f"Nulos em QUANTIDADE: {df_limpo['QUANTIDADE'].isnull().sum()}")

Tipos depois da correção:
QUANTIDADE    float64
PRECO_UNIT    float64
dtype: object

Nulos em PRECO_UNIT: 83
Nulos em QUANTIDADE: 131


In [8]:
# Vendo o que tinha originalmente nas linhas que viraram nulo
nulos_preco = df[df_limpo.index.isin(df_limpo[df_limpo['PRECO_UNIT'].isnull()].index)]
print("Valores originais de PRECO_UNIT que viraram nulo:")
print(df.loc[df_limpo['PRECO_UNIT'].isnull(), 'PRECO_UNIT'].unique())

print("\nValores originais de QUANTIDADE que viraram nulo:")
print(df.loc[df_limpo['QUANTIDADE'].isnull(), 'QUANTIDADE'].unique())

Valores originais de PRECO_UNIT que viraram nulo:
[nan]

Valores originais de QUANTIDADE que viraram nulo:
[nan]


In [9]:
# Sinalizando registros problemáticos em vez de remover
df_limpo['PRECO_UNIT_STATUS'] = df_limpo['PRECO_UNIT'].apply(
    lambda x: 'OK' if pd.notna(x) and x > 0 else 'REVISAR'
)

print(df_limpo['PRECO_UNIT_STATUS'].value_counts())
print("\nRegistros que precisam de revisão:")
print(df_limpo[df_limpo['PRECO_UNIT_STATUS'] == 'REVISAR'][['ID_VENDA', 'PRODUTO', 'VENDEDOR', 'NOTA_FISCAL']].head(10))

PRECO_UNIT_STATUS
OK         217
REVISAR     83
Name: count, dtype: int64

Registros que precisam de revisão:
    ID_VENDA                    PRODUTO      VENDEDOR  NOTA_FISCAL
2          3       geladeira frost free   João Santos      91070.0
3          4            micro-ondas 30l  Carlos Silva      97841.0
5          6            micro-ondas 30l   João Santos      80644.0
7          8           micro ondas 30 l    Maria Lima      10425.0
10        11  ar condicionado 12000 btu   Pedro Costa      77839.0
13        14             sofa 3 lugares   João Santos      28373.0
21        22              televisao 55'     Ana Souza          NaN
26        27           micro ondas 30 l   Pedro Costa      91678.0
27        28            micro-ondas 30l     Ana Souza      59695.0
32        33   ar condicionado 12000btu     Ana Souza      20155.0


In [10]:
# Separando os dados limpos dos problemáticos
df_ok = df_limpo[df_limpo['PRECO_UNIT_STATUS'] == 'OK'].copy()
df_revisar = df_limpo[df_limpo['PRECO_UNIT_STATUS'] == 'REVISAR'].copy()

# Exportando
df_ok.to_excel('vendas_LIMPO.xlsx', index=False)
df_revisar.to_excel('vendas_REVISAR.xlsx', index=False)

print(f"Arquivo limpo: {len(df_ok)} registros")
print(f"Arquivo para revisão: {len(df_revisar)} registros")
print("Arquivos salvos!")

Arquivo limpo: 217 registros
Arquivo para revisão: 83 registros
Arquivos salvos!


In [11]:
print("=" * 50)
print("RELATÓRIO DE LIMPEZA DE DADOS")
print("=" * 50)

print(f"\n📊 DADOS ORIGINAIS:")
print(f"   Total de registros: {len(df)}")
print(f"   Meses: Janeiro e Fevereiro/2025")

print(f"\n🔧 PROBLEMAS ENCONTRADOS E TRATADOS:")
print(f"   • Vendedores: {df['VENDEDOR'].nunique()} versões → {df_limpo['VENDEDOR'].nunique()} vendedores únicos")
print(f"   • Cidades: {df['CIDADE'].nunique()} versões → {df_limpo['CIDADE'].nunique()} cidades únicas")
print(f"   • PRECO_UNIT: estava como texto, convertido para número")
print(f"   • QUANTIDADE: estava como texto com ' un', convertido para número")
print(f"   • Erros de digitação corrigidos: Salavdor, Curityba, Perdo Costa, etc")
print(f"   • Formatos inconsistentes padronizados: R$, vírgula, caixa alta/baixa")

print(f"\n✅ RESULTADO:")
print(f"   Registros limpos prontos para análise: {len(df_ok)}")
print(f"   Registros sinalizados para revisão: {len(df_revisar)}")

print(f"\n📁 ARQUIVOS GERADOS:")
print(f"   vendas_LIMPO.xlsx — pronto para Power BI")
print(f"   vendas_REVISAR.xlsx — para revisão com o time")
print("=" * 50)

RELATÓRIO DE LIMPEZA DE DADOS

📊 DADOS ORIGINAIS:
   Total de registros: 300
   Meses: Janeiro e Fevereiro/2025

🔧 PROBLEMAS ENCONTRADOS E TRATADOS:
   • Vendedores: 25 versões → 5 vendedores únicos
   • Cidades: 25 versões → 5 cidades únicas
   • PRECO_UNIT: estava como texto, convertido para número
   • QUANTIDADE: estava como texto com ' un', convertido para número
   • Erros de digitação corrigidos: Salavdor, Curityba, Perdo Costa, etc
   • Formatos inconsistentes padronizados: R$, vírgula, caixa alta/baixa

✅ RESULTADO:
   Registros limpos prontos para análise: 217
   Registros sinalizados para revisão: 83

📁 ARQUIVOS GERADOS:
   vendas_LIMPO.xlsx — pronto para Power BI
   vendas_REVISAR.xlsx — para revisão com o time


In [12]:
print("PAGAMENTO único:")
print(sorted(df_limpo['PAGAMENTO'].unique()))

print("\nCATEGORIA única:")
print(sorted(df_limpo['CATEGORIA'].unique()))

print("\nPRODUTO único:")
print(sorted(df_limpo['PRODUTO'].unique()))

PAGAMENTO único:
['boleto', 'boleto bancario', 'boleto bancário', 'cartao de credito', 'cartao de debito', 'cartão credito', 'cartão de crédito', 'cartão de débito', 'cartão debito', 'credito', 'debito', 'dinheiro', 'especie', 'espécie', 'pix']

CATEGORIA única:
['celular', 'celulares', 'climatizacao', 'climatização', 'electrodomestico', 'eletrodomestico', 'eletrodoméstico', 'eletronico', 'eletrônico', 'informatica', 'informática', 'linha branca', 'mobilia', 'mobília', 'moveis', 'móveis', 'telefonia']

PRODUTO único:
['ar condicionado 12000 btu', 'ar condicionado 12000btu', 'ar-condicionado 12000btu', 'arcondicionado 12000 btu', 'celular  samsung', 'celular samsumg', 'celular samsung', 'geladeira frost free', 'geladeira frost-free', 'geladeira frostfree', 'micro ondas 30 l', 'micro ondas 30l', 'micro-ondas 30l', 'microondas 30l', 'note book gamer', 'notebbok gamer', 'notebook gamer', 'sofa 3 lugares', 'sofá 3 lugares', 'sófa 3 lugares', 'televisao 55', "televisao 55'", "televisão 55'",

In [13]:
# CORRIGINDO PAGAMENTO
correcoes_pagamento = {
    'boleto bancario': 'boleto',
    'boleto bancário': 'boleto',
    'cartao de credito': 'cartão de crédito',
    'cartao de debito': 'cartão de débito',
    'cartão credito': 'cartão de crédito',
    'cartão debito': 'cartão de débito',
    'credito': 'cartão de crédito',
    'debito': 'cartão de débito',
    'debido': 'cartão de débito',
    'cartao debido': 'cartão de débito',
    'cartão de débito': 'cartão de débito',
    'especie': 'dinheiro',
    'espécie': 'dinheiro',
}
df_limpo['PAGAMENTO'] = df_limpo['PAGAMENTO'].replace(correcoes_pagamento)

# CORRIGINDO CATEGORIA
correcoes_categoria = {
    'celulares': 'celular',
    'climatizacao': 'climatização',
    'electrodomestico': 'eletrodoméstico',
    'eletrodomestico': 'eletrodoméstico',
    'eletronico': 'eletrônico',
    'informatica': 'informática',
    'linha branca': 'eletrodoméstico',
    'mobilia': 'móveis',
    'moveis': 'móveis',
    'telefonia': 'celular',
}
df_limpo['CATEGORIA'] = df_limpo['CATEGORIA'].replace(correcoes_categoria)

# CORRIGINDO PRODUTO
correcoes_produto = {
    'ar condicionado 12000 btu': 'ar condicionado 12000btu',
    'ar condicionado 12000btu': 'ar condicionado 12000btu',
    'ar-condicionado 12000btu': 'ar condicionado 12000btu',
    'arcondicionado 12000 btu': 'ar condicionado 12000btu',
    'celular  samsung': 'celular samsung',
    'celular samsumg': 'celular samsung',
    'geladeira frost-free': 'geladeira frost free',
    'geladeira frostfree': 'geladeira frost free',
    'micro ondas 30 l': 'micro-ondas 30l',
    'micro ondas 301': 'micro-ondas 30l',
    'microondas 301': 'micro-ondas 30l',
    'microondas 30l': 'micro-ondas 30l',
    'note book gamer': 'notebook gamer',
    'notebbok gamer': 'notebook gamer',
    'sofa 3 lugares': 'sofá 3 lugares',
    'sófa 3 lugares': 'sofá 3 lugares',
    'televisao 55': 'televisão 55\'',
    'televisao 55\'': 'televisão 55\'',
    'tv 55 polegadas': 'televisão 55\'',
    'tv 55\'': 'televisão 55\'',
}
df_limpo['PRODUTO'] = df_limpo['PRODUTO'].replace(correcoes_produto)

# VERIFICANDO RESULTADO
print("PAGAMENTO:", sorted(df_limpo['PAGAMENTO'].unique()))
print("\nCATEGORIA:", sorted(df_limpo['CATEGORIA'].unique()))
print("\nPRODUTO:", sorted(df_limpo['PRODUTO'].unique()))

PAGAMENTO: ['boleto', 'cartão de crédito', 'cartão de débito', 'dinheiro', 'pix']

CATEGORIA: ['celular', 'climatização', 'eletrodoméstico', 'eletrônico', 'informática', 'mobília', 'móveis']

PRODUTO: ['ar condicionado 12000btu', 'celular samsung', 'geladeira frost free', 'micro ondas 30l', 'micro-ondas 30l', 'notebook gamer', 'sofá 3 lugares', "televisão 55'"]


In [14]:
# Corrigindo último detalhe do produto
correcoes_final = {
    'micro ondas 301': 'micro-ondas 30l',
}
df_limpo['PRODUTO'] = df_limpo['PRODUTO'].replace(correcoes_final)

print("PRODUTO:", sorted(df_limpo['PRODUTO'].unique()))

PRODUTO: ['ar condicionado 12000btu', 'celular samsung', 'geladeira frost free', 'micro ondas 30l', 'micro-ondas 30l', 'notebook gamer', 'sofá 3 lugares', "televisão 55'"]


In [15]:
correcoes_final = {
    'micro ondas 301': 'micro-ondas 30l',
    'micro-ondas 301': 'micro-ondas 30l',
}
df_limpo['PRODUTO'] = df_limpo['PRODUTO'].replace(correcoes_final)

print("PRODUTO:", sorted(df_limpo['PRODUTO'].unique()))

PRODUTO: ['ar condicionado 12000btu', 'celular samsung', 'geladeira frost free', 'micro ondas 30l', 'micro-ondas 30l', 'notebook gamer', 'sofá 3 lugares', "televisão 55'"]


In [16]:
df_limpo['PRODUTO'] = df_limpo['PRODUTO'].str.replace('micro ondas 301', 'micro-ondas 30l', regex=False)
df_limpo['PRODUTO'] = df_limpo['PRODUTO'].str.replace('micro-ondas 301', 'micro-ondas 30l', regex=False)

print("PRODUTO:", sorted(df_limpo['PRODUTO'].unique()))

PRODUTO: ['ar condicionado 12000btu', 'celular samsung', 'geladeira frost free', 'micro ondas 30l', 'micro-ondas 30l', 'notebook gamer', 'sofá 3 lugares', "televisão 55'"]


In [17]:
df_limpo['PRODUTO'] = df_limpo['PRODUTO'].str.replace('micro ondas 301', 'micro-ondas 30l', regex=False)
df_limpo['PRODUTO'] = df_limpo['PRODUTO'].str.replace('micro-ondas 301', 'micro-ondas 30l', regex=False)

print("PRODUTO:", sorted(df_limpo['PRODUTO'].unique()))

PRODUTO: ['ar condicionado 12000btu', 'celular samsung', 'geladeira frost free', 'micro ondas 30l', 'micro-ondas 30l', 'notebook gamer', 'sofá 3 lugares', "televisão 55'"]


In [18]:
# Verificando o que realmente tem no dado
for p in df_limpo['PRODUTO'].unique():
    if 'micro' in p:
        print(repr(p))

'micro-ondas 30l'
'micro ondas 30l'


In [19]:
df_limpo['PRODUTO'] = df_limpo['PRODUTO'].str.replace('micro-ondas 301', 'micro-ondas 30l', regex=False)
df_limpo['PRODUTO'] = df_limpo['PRODUTO'].str.replace('micro ondas 301', 'micro-ondas 30l', regex=False)

for p in df_limpo['PRODUTO'].unique():
    if 'micro' in p:
        print(repr(p))

'micro-ondas 30l'
'micro ondas 30l'


In [20]:
for p in df_limpo['PRODUTO'].unique():
    if 'micro' in p:
        print(repr(p))
        print([ord(c) for c in p])  # mostra o código de cada caractere

'micro-ondas 30l'
[109, 105, 99, 114, 111, 45, 111, 110, 100, 97, 115, 32, 51, 48, 108]
'micro ondas 30l'
[109, 105, 99, 114, 111, 32, 111, 110, 100, 97, 115, 32, 51, 48, 108]


In [21]:
df_limpo['PRODUTO'] = df_limpo['PRODUTO'].str.replace('micro ondas 30l', 'micro-ondas 30l', regex=False)

for p in df_limpo['PRODUTO'].unique():
    if 'micro' in p:
        print(repr(p))

'micro-ondas 30l'


In [22]:
print("PAGAMENTO:", sorted(df_limpo['PAGAMENTO'].unique()))
print("\nCATEGORIA:", sorted(df_limpo['CATEGORIA'].unique()))
print("\nPRODUTO:", sorted(df_limpo['PRODUTO'].unique()))
print("\nVENDEDOR:", sorted(df_limpo['VENDEDOR'].unique()))
print("\nCIDADE:", sorted(df_limpo['CIDADE'].unique()))
print(f"\nNulos restantes:")
print(df_limpo.isnull().sum())

PAGAMENTO: ['boleto', 'cartão de crédito', 'cartão de débito', 'dinheiro', 'pix']

CATEGORIA: ['celular', 'climatização', 'eletrodoméstico', 'eletrônico', 'informática', 'mobília', 'móveis']

PRODUTO: ['ar condicionado 12000btu', 'celular samsung', 'geladeira frost free', 'micro-ondas 30l', 'notebook gamer', 'sofá 3 lugares', "televisão 55'"]

VENDEDOR: ['Ana Souza', 'Carlos Silva', 'João Santos', 'Maria Lima', 'Pedro Costa']

CIDADE: ['Belo Horizonte', 'Curitiba', 'Rio De Janeiro', 'Salvador', 'São Paulo']

Nulos restantes:
ID_VENDA               0
DATA_VENDA            43
PRODUTO                0
CATEGORIA              0
QUANTIDADE           131
PRECO_UNIT            83
VENDEDOR               0
PAGAMENTO              0
CIDADE                 0
CPF_CLIENTE           57
NOTA_FISCAL           45
PRECO_UNIT_STATUS      0
dtype: int64


In [23]:
def classificar_registro(row):
    problemas = []
    
    if pd.isna(row['DATA_VENDA']):
        problemas.append('SEM DATA')
    
    if pd.isna(row['QUANTIDADE']) or row['QUANTIDADE'] == 0:
        problemas.append('SEM QUANTIDADE')
    
    if pd.isna(row['NOTA_FISCAL']):
        problemas.append('SEM NOTA FISCAL')
    
    if row['PRECO_UNIT_STATUS'] == 'REVISAR':
        problemas.append('SEM PRECO')
    
    if problemas:
        return ' | '.join(problemas)
    return 'OK'

df_limpo['STATUS_GERAL'] = df_limpo.apply(classificar_registro, axis=1)

print(df_limpo['STATUS_GERAL'].value_counts())

STATUS_GERAL
OK                                                         95
SEM QUANTIDADE                                             66
SEM PRECO                                                  32
SEM QUANTIDADE | SEM PRECO                                 28
SEM DATA | SEM QUANTIDADE                                  14
SEM QUANTIDADE | SEM NOTA FISCAL                           13
SEM DATA                                                   11
SEM NOTA FISCAL                                            11
SEM DATA | SEM NOTA FISCAL                                  7
SEM NOTA FISCAL | SEM PRECO                                 7
SEM DATA | SEM PRECO                                        5
SEM QUANTIDADE | SEM NOTA FISCAL | SEM PRECO                5
SEM DATA | SEM QUANTIDADE | SEM PRECO                       4
SEM DATA | SEM NOTA FISCAL | SEM PRECO                      1
SEM DATA | SEM QUANTIDADE | SEM NOTA FISCAL | SEM PRECO     1
Name: count, dtype: int64


In [24]:
print("Formatos de data encontrados:")
print(df_limpo['DATA_VENDA'].dropna().unique()[:20])

Formatos de data encontrados:
<StringArray>
[ '2025-01-08', 'jan/21/2025',  '03/01/2025',  '06.01.2025',  '07.01.2025',
  '01/19/2025',  '15.01.2025',  '17/01/2025',  '03-01-2025',  '2025-01-16',
  '01/06/2025',  '18/01/2025',  '2025-01-28', 'jan/07/2025',  '17-01-2025',
  '05-01-2025',  '27-01-2025',  '12.01.2025',  '01/01/2025', 'jan/02/2025']
Length: 20, dtype: str


In [25]:
from dateutil import parser

def converter_data(data):
    if pd.isna(data) or data == 'nan':
        return None
    try:
        return parser.parse(str(data), dayfirst=True)
    except:
        return None

df_limpo['DATA_VENDA'] = df_limpo['DATA_VENDA'].apply(converter_data)

print("Nulos após conversão:", df_limpo['DATA_VENDA'].isnull().sum())
print("\nAmostras:")
print(df_limpo['DATA_VENDA'].dropna().head(10))

Nulos após conversão: 57

Amostras:
0    2025-08-01
1    2025-01-21
3    2025-01-03
4    2025-01-06
5    2025-01-07
6    2025-01-19
7    2025-01-15
8    2025-01-17
11   2025-01-03
12   2025-01-16
Name: DATA_VENDA, dtype: datetime64[us]


In [26]:
# Verificando datas fora do período esperado
datas_suspeitas = df_limpo[
    (df_limpo['DATA_VENDA'].notna()) & 
    (df_limpo['DATA_VENDA'].dt.month.isin([3,4,5,6,7,8,9,10,11,12]))
]

print(f"Datas fora de jan/fev: {len(datas_suspeitas)}")
print(datas_suspeitas[['ID_VENDA', 'DATA_VENDA']].head(10))

Datas fora de jan/fev: 31
     ID_VENDA DATA_VENDA
0           1 2025-08-01
15         16 2025-06-01
24         25 2025-08-01
49         50 2025-06-01
57         58 2025-05-01
77         78 2025-08-01
119       120 2025-08-01
145       146 2025-05-01
149       150 2025-08-01
150       151 2025-06-02


In [27]:
# Marcando datas suspeitas
df_limpo['DATA_STATUS'] = df_limpo['DATA_VENDA'].apply(
    lambda x: 'REVISAR' if pd.notna(x) and x.month not in [1, 2] else (
        'NULA' if pd.isna(x) else 'OK'
    )
)

print(df_limpo['DATA_STATUS'].value_counts())

DATA_STATUS
OK         212
NULA        57
REVISAR     31
Name: count, dtype: int64


In [28]:
# Verificando quantidade por produto
print("=== ZEROS POR PRODUTO ===")
zeros = df_limpo[df_limpo['QUANTIDADE'] == 0]
print(zeros['PRODUTO'].value_counts())

print("\n=== NULOS POR PRODUTO ===")
nulos_qtd = df_limpo[df_limpo['QUANTIDADE'].isna()]
print(nulos_qtd['PRODUTO'].value_counts())

print("\n=== QUANTIDADE MÉDIA POR PRODUTO (registros válidos) ===")
print(df_limpo[df_limpo['QUANTIDADE'] > 0].groupby('PRODUTO')['QUANTIDADE'].mean().round(1))

=== ZEROS POR PRODUTO ===
Series([], Name: count, dtype: int64)

=== NULOS POR PRODUTO ===
PRODUTO
notebook gamer              26
micro-ondas 30l             21
geladeira frost free        20
televisão 55'               19
celular samsung             19
ar condicionado 12000btu    15
sofá 3 lugares              11
Name: count, dtype: int64

=== QUANTIDADE MÉDIA POR PRODUTO (registros válidos) ===
PRODUTO
ar condicionado 12000btu    3.8
celular samsung             2.6
geladeira frost free        2.7
micro-ondas 30l             2.8
notebook gamer              2.8
sofá 3 lugares              2.9
televisão 55'               2.7
Name: QUANTIDADE, dtype: float64


In [29]:
# Preenchendo nulos de QUANTIDADE com a média do produto
media_por_produto = df_limpo[df_limpo['QUANTIDADE'] > 0].groupby('PRODUTO')['QUANTIDADE'].mean()

def preencher_quantidade(row):
    if pd.isna(row['QUANTIDADE']):
        media = media_por_produto.get(row['PRODUTO'], None)
        if media:
            return round(media)  # arredonda pra inteiro
    return row['QUANTIDADE']

df_limpo['QUANTIDADE'] = df_limpo.apply(preencher_quantidade, axis=1)
df_limpo['QUANTIDADE'] = df_limpo['QUANTIDADE'].astype(int)

print(f"Nulos restantes em QUANTIDADE: {df_limpo['QUANTIDADE'].isnull().sum()}")
print(f"\nMédia por produto após preenchimento:")
print(df_limpo.groupby('PRODUTO')['QUANTIDADE'].mean().round(1))

Nulos restantes em QUANTIDADE: 0

Média por produto após preenchimento:
PRODUTO
ar condicionado 12000btu    3.9
celular samsung             2.8
geladeira frost free        2.8
micro-ondas 30l             2.9
notebook gamer              2.9
sofá 3 lugares              2.9
televisão 55'               2.8
Name: QUANTIDADE, dtype: float64


In [30]:
# NOTA_FISCAL para inteiro
df_limpo['NOTA_FISCAL'] = df_limpo['NOTA_FISCAL'].fillna(0).astype(int)

# CPF — padronizar nulos como 'NÃO INFORMADO'
df_limpo['CPF_CLIENTE'] = df_limpo['CPF_CLIENTE'].fillna('NÃO INFORMADO')

# Verificando
print(f"Nulos em NOTA_FISCAL: {df_limpo['NOTA_FISCAL'].isnull().sum()}")
print(f"Tipo NOTA_FISCAL: {df_limpo['NOTA_FISCAL'].dtype}")
print(f"\nNulos em CPF_CLIENTE: {df_limpo['CPF_CLIENTE'].isnull().sum()}")
print(f"\nCPF únicos:")
print(df_limpo['CPF_CLIENTE'].value_counts().tail(3))

Nulos em NOTA_FISCAL: 0
Tipo NOTA_FISCAL: int64

Nulos em CPF_CLIENTE: 0

CPF únicos:
CPF_CLIENTE
194.546.275-14    1
727.801.512-32    1
949.306.912-97    1
Name: count, dtype: int64


In [31]:
print("=== CHECAGEM FINAL ===")
print(f"\nTipos de dados:")
print(df_limpo.dtypes)
print(f"\nNulos restantes:")
print(df_limpo.isnull().sum())
print(f"\nValores únicos por coluna:")
print(f"PRODUTO: {sorted(df_limpo['PRODUTO'].unique())}")
print(f"CATEGORIA: {sorted(df_limpo['CATEGORIA'].unique())}")
print(f"VENDEDOR: {sorted(df_limpo['VENDEDOR'].unique())}")
print(f"PAGAMENTO: {sorted(df_limpo['PAGAMENTO'].unique())}")
print(f"CIDADE: {sorted(df_limpo['CIDADE'].unique())}")
print(f"\nPRECO_UNIT — min: {df_limpo['PRECO_UNIT'].min()}, max: {df_limpo['PRECO_UNIT'].max()}")
print(f"QUANTIDADE — min: {df_limpo['QUANTIDADE'].min()}, max: {df_limpo['QUANTIDADE'].max()}")

=== CHECAGEM FINAL ===

Tipos de dados:
ID_VENDA                      int64
DATA_VENDA           datetime64[us]
PRODUTO                         str
CATEGORIA                       str
QUANTIDADE                    int64
PRECO_UNIT                  float64
VENDEDOR                        str
PAGAMENTO                       str
CIDADE                          str
CPF_CLIENTE                     str
NOTA_FISCAL                   int64
PRECO_UNIT_STATUS               str
STATUS_GERAL                    str
DATA_STATUS                     str
dtype: object

Nulos restantes:
ID_VENDA              0
DATA_VENDA           57
PRODUTO               0
CATEGORIA             0
QUANTIDADE            0
PRECO_UNIT           83
VENDEDOR              0
PAGAMENTO             0
CIDADE                0
CPF_CLIENTE           0
NOTA_FISCAL           0
PRECO_UNIT_STATUS     0
STATUS_GERAL          0
DATA_STATUS           0
dtype: int64

Valores únicos por coluna:
PRODUTO: ['ar condicionado 12000btu', 'celular 

In [32]:
# Forçando correção direta nos valores
df_limpo.loc[df_limpo['PRODUTO'] == 'micro-ondas 301', 'PRODUTO'] = 'micro-ondas 30l'

print("PRODUTO:", sorted(df_limpo['PRODUTO'].unique()))

PRODUTO: ['ar condicionado 12000btu', 'celular samsung', 'geladeira frost free', 'micro-ondas 30l', 'notebook gamer', 'sofá 3 lugares', "televisão 55'"]


In [33]:
df_limpo.loc[df_limpo['PRODUTO'] == 'micro-ondas 301', 'PRODUTO'] = 'micro-ondas 30L'

print("PRODUTO:", sorted(df_limpo['PRODUTO'].unique()))

PRODUTO: ['ar condicionado 12000btu', 'celular samsung', 'geladeira frost free', 'micro-ondas 30l', 'notebook gamer', 'sofá 3 lugares', "televisão 55'"]


In [34]:
# Atualizando STATUS_GERAL final
def status_final(row):
    problemas = []
    
    if row['DATA_STATUS'] == 'NULA':
        problemas.append('SEM DATA')
    elif row['DATA_STATUS'] == 'REVISAR':
        problemas.append('DATA SUSPEITA')
    
    if row['NOTA_FISCAL'] == 0:
        problemas.append('SEM NOTA FISCAL')
    
    if row['PRECO_UNIT_STATUS'] == 'REVISAR':
        problemas.append('SEM PRECO')
    
    return ' | '.join(problemas) if problemas else 'OK'

df_limpo['STATUS_GERAL'] = df_limpo.apply(status_final, axis=1)

# Separando
df_ok_final = df_limpo[df_limpo['STATUS_GERAL'] == 'OK'].copy()
df_problemas_final = df_limpo[df_limpo['STATUS_GERAL'] != 'OK'].copy()

# Exportando
df_ok_final.to_excel('vendas_LIMPO_FINAL.xlsx', index=False)
df_problemas_final.to_excel('vendas_PROBLEMAS_FINAL.xlsx', index=False)

print("=== RELATÓRIO FINAL ===")
print(f"Total de registros: {len(df_limpo)}")
print(f"Registros OK: {len(df_ok_final)}")
print(f"Registros com problemas: {len(df_problemas_final)}")
print(f"\nDetalhes dos problemas:")
print(df_limpo['STATUS_GERAL'].value_counts())
print("\n✅ Arquivos exportados!")

=== RELATÓRIO FINAL ===
Total de registros: 300
Registros OK: 135
Registros com problemas: 165

Detalhes dos problemas:
STATUS_GERAL
OK                                             135
SEM PRECO                                       48
SEM DATA                                        35
SEM NOTA FISCAL                                 22
DATA SUSPEITA                                   16
SEM DATA | SEM PRECO                            11
DATA SUSPEITA | SEM PRECO                       10
SEM DATA | SEM NOTA FISCAL                       7
SEM NOTA FISCAL | SEM PRECO                      7
SEM DATA | SEM NOTA FISCAL | SEM PRECO           4
DATA SUSPEITA | SEM NOTA FISCAL | SEM PRECO      3
DATA SUSPEITA | SEM NOTA FISCAL                  2
Name: count, dtype: int64

✅ Arquivos exportados!


In [35]:
print("=== ANÁLISE EXPLORATÓRIA ===")

print("\n📦 VENDAS POR PRODUTO:")
print(df_ok_final.groupby('PRODUTO')['ID_VENDA'].count().sort_values(ascending=False))

print("\n💰 FATURAMENTO POR PRODUTO:")
df_ok_final['FATURAMENTO'] = df_ok_final['QUANTIDADE'] * df_ok_final['PRECO_UNIT']
print(df_ok_final.groupby('PRODUTO')['FATURAMENTO'].sum().sort_values(ascending=False).round(2))

print("\n👤 FATURAMENTO POR VENDEDOR:")
print(df_ok_final.groupby('VENDEDOR')['FATURAMENTO'].sum().sort_values(ascending=False).round(2))

print("\n🏙️ FATURAMENTO POR CIDADE:")
print(df_ok_final.groupby('CIDADE')['FATURAMENTO'].sum().sort_values(ascending=False).round(2))

print("\n💳 VENDAS POR FORMA DE PAGAMENTO:")
print(df_ok_final.groupby('PAGAMENTO')['ID_VENDA'].count().sort_values(ascending=False))

print("\n📅 FATURAMENTO POR MÊS:")
df_ok_final['MES'] = df_ok_final['DATA_VENDA'].dt.month
print(df_ok_final.groupby('MES')['FATURAMENTO'].sum().round(2))

=== ANÁLISE EXPLORATÓRIA ===

📦 VENDAS POR PRODUTO:
PRODUTO
geladeira frost free        25
celular samsung             23
micro-ondas 30l             21
televisão 55'               20
notebook gamer              19
sofá 3 lugares              15
ar condicionado 12000btu    12
Name: ID_VENDA, dtype: int64

💰 FATURAMENTO POR PRODUTO:
PRODUTO
notebook gamer              255566.81
geladeira frost free        223980.23
ar condicionado 12000btu    198305.96
televisão 55'               152824.50
celular samsung             126949.17
sofá 3 lugares              101775.51
micro-ondas 30l              35995.63
Name: FATURAMENTO, dtype: float64

👤 FATURAMENTO POR VENDEDOR:
VENDEDOR
Ana Souza       324120.53
Maria Lima      237222.55
Carlos Silva    209430.27
João Santos     169061.76
Pedro Costa     155562.70
Name: FATURAMENTO, dtype: float64

🏙️ FATURAMENTO POR CIDADE:
CIDADE
Belo Horizonte    262508.74
São Paulo         260518.10
Salvador          252418.27
Rio De Janeiro    207162.98
Curitiba 

In [36]:
print("=== ANÁLISE POR VENDEDOR ===")

# Quais produtos cada vendedor vendeu
print("\n📦 PRODUTOS POR VENDEDOR:")
print(df_ok_final.groupby(['VENDEDOR', 'PRODUTO'])['ID_VENDA'].count().unstack(fill_value=0))

print("\n💰 TICKET MÉDIO POR VENDEDOR:")
print(df_ok_final.groupby('VENDEDOR')['PRECO_UNIT'].mean().sort_values(ascending=False).round(2))

print("\n🔢 QUANTIDADE MÉDIA VENDIDA POR VENDEDOR:")
print(df_ok_final.groupby('VENDEDOR')['QUANTIDADE'].mean().sort_values(ascending=False).round(1))

print("\n📊 VENDAS POR VENDEDOR:")
vendedor_resumo = df_ok_final.groupby('VENDEDOR').agg(
    Total_Vendas=('ID_VENDA', 'count'),
    Faturamento=('FATURAMENTO', 'sum'),
    Ticket_Medio=('PRECO_UNIT', 'mean'),
    Qtd_Media=('QUANTIDADE', 'mean')
).round(2)
print(vendedor_resumo.sort_values('Faturamento', ascending=False))

=== ANÁLISE POR VENDEDOR ===

📦 PRODUTOS POR VENDEDOR:
PRODUTO       ar condicionado 12000btu  celular samsung  geladeira frost free  \
VENDEDOR                                                                        
Ana Souza                            6                5                     7   
Carlos Silva                         1                4                     3   
João Santos                          0                8                     5   
Maria Lima                           3                3                     6   
Pedro Costa                          2                3                     4   

PRODUTO       micro-ondas 30l  notebook gamer  sofá 3 lugares  televisão 55'  
VENDEDOR                                                                      
Ana Souza                   5               5               5              5  
Carlos Silva                5               4               5              5  
João Santos                 5               3               3